---
title: "Building Real-World AI Systems with PydanticAI: Lessons from Anthropic's Agent Design Patterns"
author: "Fabian"
date: "2025-05-21"
categories: [Agents, Pydantic]
format:
  html:
    code-fold: false
    toc: true
    echo: true
jupyter: python3
freeze: true

---

# Introduction: Why Agent Design Patterns Matter
As AI continues to reshape industries—from legal and healthcare to marketing and customer service—developers face a growing need to build robust, modular, and intelligent systems. This is where agent design patterns come into play.

In a pivotal post titled Building Effective Agents, Anthropic introduced core agent patterns that help AI engineers build complex yet maintainable systems. These include strategies like prompt chaining, routing, parallel execution, orchestration, and optimization.

In this post, we'll translate these patterns into real-world applications using PydanticAI—a framework that makes it easy to define and run structured AI agents. We'll walk through five major patterns and how they apply to real use cases, with fully working code samples.

https://www.anthropic.com/engineering/building-effective-agents

![Augmenting LLM's](augmented_llm.png)

## 1. Prompt Chaining – Use Case: Legal Document Review Workflow
Scenario: Automating the extraction, summarization, and risk analysis of legal documents.

In [1]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIModel 

import nest_asyncio
nest_asyncio.apply()

# Load environment variables from .env file
load_dotenv()

# Get the OpenAI API key from the environment variable
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")  

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY environment variable not set.")

# Initialize our LLM model
model = OpenAIModel( 
    model_name='gpt-4o-mini', 
    api_key=OPENAI_API_KEY,
)


class ExtractedClauses(BaseModel):
    clauses: list[str]

class Summary(BaseModel):
    summary: str

class RiskAssessment(BaseModel):
    risk_level: str
    issues: list[str]

extractor = Agent(
    model="openai:gpt-4",
    result_type=ExtractedClauses,
    system_prompt="Extract key clauses from the legal document."
)

summarizer = Agent(
    model="openai:gpt-4",
    result_type=Summary,
    system_prompt="Summarize the extracted legal clauses."
)

risk_analyzer = Agent(
    model="openai:gpt-4",
    result_type=RiskAssessment,
    system_prompt="Assess potential legal risks in the summary."
)

# Read the terms file from open ai copied into a txt file. http://openai.com/policies/row-terms-of-use/
with open('./openai-terms.txt', 'r') as file:
    text = file.read()
print(text[0:100])
clauses = extractor.run_sync(text)
summary = summarizer.run_sync(str(clauses.data.clauses))
risk = risk_analyzer.run_sync(summary.data.summary)

print(clauses.data)
print(summary.data)
print(risk.data)


Thank you for using OpenAI!

These Terms of Use apply to your use of ChatGPT, DALL·E, and OpenAI’s o
clauses=['These Terms of Use apply to your use of ChatGPT, DALL·E, and OpenAI’s other services for individuals, along with any associated software applications and websites (all together, “Services”).', 'These Terms form an agreement between you and OpenAI, L.L.C., a Delaware company, and they include our Service Terms\u2060 and important provisions for resolving disputes through arbitration.', 'By using our Services, you agree to these Terms.', 'Our Business Terms\u2060 govern use of ChatGPT Enterprise, our APIs, and our other services for businesses and developers.', 'Our Privacy Policy\u2060 explains how we collect and use personal information.', 'You must provide accurate and complete information to register for an account to use our Services.', 'You represent and warrant that you have all rights, licenses, and permissions needed to provide Input to our Services.', 'We may use Conte

## 2. Routing – Use Case: Customer Support Agent
Scenario: Route customer inquiries to specialized support agents (billing, technical, general).

In [2]:
billing_agent = Agent(model=model,
                    system_prompt="Answer billing questions."
                       )
tech_agent = Agent(model=model,
                    system_prompt="Handle technical support.")
general_agent = Agent(model=model,
                    system_prompt="Answer general customer questions.")

def route_query(input_text: str):
    if "bill" in input_text.lower():
        return billing_agent.run_sync(input_text)
    elif "error" in input_text.lower():
        return tech_agent.run_sync(input_text)
    else:
        return general_agent.run_sync(input_text)

response = route_query("I have a problem with my last invoice.")
print(response.data)


I'm sorry to hear that you're having an issue with your invoice. Could you please provide more details about the problem? For example, are there discrepancies in the amount, missing charges, or any other specific concerns? This will help me assist you better.


## 3. Parallelization – Use Case: Market Intelligence Dashboard
Scenario: Gather insights on competitors, customer reviews, and product trends concurrently.

In [ ]:
import asyncio
from pydantic_ai.common_tools.duckduckgo import duckduckgo_search_tool

competitor_agent = Agent(
    model=model,
    tools=[duckduckgo_search_tool()],
    system_prompt='Search DuckDuckGo for the given query and return the results. Summarize competitor activity.',
)

reviews_agent = Agent(
    model=model,
    tools=[duckduckgo_search_tool()],
    system_prompt='Search DuckDuckGo for the given query and return the results. Summarize customer reviews.',
)

trends_agent = Agent(
    model=model,
    tools=[duckduckgo_search_tool()],
    system_prompt='Search DuckDuckGo for the given query and return the results. Summarize industry trends.',
)

async def collect_insights():
    competitor_task = asyncio.create_task(competitor_agent.run("Company A"))
    reviews_task = asyncio.create_task(reviews_agent.run("Product XYZ"))
    trends_task = asyncio.create_task(trends_agent.run("Industry trends"))

    insights = await asyncio.gather(competitor_task, reviews_task, trends_task)
    return insights

insights = asyncio.run(collect_insights())
for insight in insights:
    print(insight.data)


## 4. Orchestrator-Worker – Use Case: Content Generation Pipeline
Scenario: An orchestrator agent creates a blog post outline, then worker agents write sections.

In [4]:
from pydantic import BaseModel
from typing import List

class Section(BaseModel):
    title: str
    description: str

class FullPost(BaseModel):
    title: str
    sections: List[Section]

outline_agent = Agent(
    model=model,
    result_type=FullPost,
    system_prompt="""
    You are a content strategist. Given a blog topic, outline the title and 3-5 key sections.
    Each section should include a title and a short description of what it should cover.
    Return the output in JSON format matching the FullPost schema.
    """
)

class WrittenSection(BaseModel):
    title: str
    content: str

writer_agent = Agent(
    model=model,
    result_type=WrittenSection,
    system_prompt="""
    You are a blog writer. Expand the following blog section.
    Return a well-written paragraph or two based on the given title and description.
    Output in JSON format with the section title and full content.
    """
)

def generate_blog_post(topic: str):
    # Step 1: Get outline from orchestrator
    outline = outline_agent.run_sync(topic)
    post_title = outline.data.title
    section_plans = outline.data.sections

    # Step 2: Generate each section via writer agent
    final_sections = []
    for section in section_plans:
        section_input = f"Title: {section.title}\nDescription: {section.description}"
        written = writer_agent.run_sync(section_input)
        final_sections.append({
            "title": written.data.title,
            "content": written.data.content
        })

    return {
        "title": post_title,
        "sections": final_sections
    }

# Example usage
post = generate_blog_post("The Future of AI in Personalized Education")

print(f"Title: {post['title']}")
for sec in post["sections"]:
    print(f"\n### {sec['title']}\n{sec['content']}")


Title: The Future of AI in Personalized Education

### Introduction to AI in Education
Artificial Intelligence (AI) is revolutionizing the landscape of education by providing tailored learning experiences that cater specifically to the individual needs of each student. In various educational settings, AI is being employed to analyze data regarding students' learning patterns, preferences, and overall performance. This enables teachers and educators to create personalized lesson plans and instructional approaches, ensuring that no student is left behind in their learning journey. 

From intelligent tutoring systems that offer immediate feedback to adaptive learning platforms that adjust the curriculum based on a student’s strengths and weaknesses, AI is enhancing the overall educational experience. Moreover, AI-powered analytics tools are helping educators identify at-risk students early on, allowing for timely intervention and support. As technology continues to advance, the potential 

## 5. Evaluator-Optimizer: Best Marketing Copy Selection
Use Case: Generate multiple slogans, evaluate their impact, and select the best.

In [5]:
from pydantic import BaseModel

class Tagline(BaseModel):
    text: str
    persuasiveness: float

generator = Agent(model=model, result_type=list[str], system_prompt="Generate 5 marketing taglines.")
evaluator = Agent(model=model, result_type=Tagline, system_prompt="Evaluate the persuasiveness of this tagline.")

taglines = generator.run_sync("Launch campaign for eco-friendly bottles")
evaluated = [evaluator.run_sync(t) for t in taglines.data]
best = max(evaluated, key=lambda t: t.data.persuasiveness)

print(taglines.data)
print(evaluated)
print(f"Best Tagline: {best.data.text} (Score: {best.data.persuasiveness})")


['Sip Sustainably with Every Drop!', 'Drink Green, Live Clean!', 'Eco Bottles: Bottle Your Impact!', 'Refill, Reuse, Revive the Planet!', 'Hydration with a Conscience!']
[RunResult(_all_messages=[ModelRequest(parts=[SystemPromptPart(content='Evaluate the persuasiveness of this tagline.', dynamic_ref=None, part_kind='system-prompt'), UserPromptPart(content='Sip Sustainably with Every Drop!', timestamp=datetime.datetime(2025, 6, 1, 1, 18, 16, 99475, tzinfo=datetime.timezone.utc), part_kind='user-prompt')], kind='request'), ModelResponse(parts=[ToolCallPart(tool_name='final_result', args=ArgsJson(args_json='{"text":"Sip Sustainably with Every Drop!","persuasiveness":8}'), tool_call_id='call_ufRsb9U6feCV9W6LXppNwvxJ', part_kind='tool-call')], timestamp=datetime.datetime(2025, 6, 1, 1, 18, 15, tzinfo=datetime.timezone.utc), kind='response'), ModelRequest(parts=[ToolReturnPart(tool_name='final_result', content='Final result processed.', tool_call_id='call_ufRsb9U6feCV9W6LXppNwvxJ', timestamp